In [1]:
# !pip install huggingface_hub xarray netcdf4
# !pip install pyshtools 
# !pip install datasets
# !pip install cartopy
# !pip install h5py

In [2]:
import os, math, time, random
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.special import sph_harm
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from itertools import islice
import h5py 
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Grid, quadrature weights, real spherical harmonics basis (Phi)
def make_grid_centers(H, W):
    dlat = math.pi / H
    dlamb = 2*math.pi / W
    lats = np.linspace(-math.pi/2 + dlat/2, math.pi/2 - dlat/2, H)  # lat centers
    lons = np.linspace(0 + dlamb/2, 2*math.pi - dlamb/2, W)         # lon centers
    theta = np.pi/2 - lats  # colatitude
    lon_grid, theta_grid = np.meshgrid(lons, theta)  # shapes (H,W)
    return lats, lons, theta_grid, lon_grid, dlat, dlamb

def build_real_sph_basis_Phi(H, W, lmax):
    """
    Build real spherical harmonic basis matrix Phi (P x Q),
    where P = H*W, Q = (lmax+1)^2, using the real basis:
      m=0 -> Y_l0.real
      m>0 -> sqrt(2)*Re(Y_lm), sqrt(2)*Im(Y_lm)
    Returns:
      Phi (P, Q) float64,
      weights (P,) quadrature weights (cos(lat)*dlat*dlon),
      index_to_l (Q,) int array mapping coefficient index -> degree l
    """
    lats, lons, theta_grid, lon_grid, dlat, dlamb = make_grid_centers(H, W)
    P = H * W
    Q = (lmax + 1)**2
    Phi = np.zeros((P, Q), dtype=np.float64)
    # quadrature weights: cos(lat) * dlat * dlamb
    lat_centers = np.pi/2 - theta_grid[:, 0]   # length H
    cos_lat = np.cos(lat_centers)              # length H
    w = np.repeat(cos_lat, W) * dlat * dlamb   # flattened P
    idx = 0
    index_to_l = np.zeros(Q, dtype=np.int32)
    for l in range(0, lmax+1):
        for m in range(0, l+1):
            Y_lm = sph_harm(m, l, lon_grid, theta_grid)  # complex HxW
            if m == 0:
                Phi[:, idx] = Y_lm.real.flatten()
                index_to_l[idx] = l
                idx += 1
            else:
                Phi[:, idx] = (np.sqrt(2.0) * Y_lm.real).flatten()
                index_to_l[idx] = l
                idx += 1
                Phi[:, idx] = (np.sqrt(2.0) * Y_lm.imag).flatten()
                index_to_l[idx] = l
                idx += 1
    assert idx == Q, f"Expected Q={Q}, built {idx}"
    return Phi, w, index_to_l

# vectorized batched transforms using precomputed A (QxP) and B (PxQ)
class SHTTransforms:
    def __init__(self, Phi: np.ndarray, weights: np.ndarray, device='cpu'):
        """
        Phi: P x Q ; weights: P  (numpy)
        Precompute A = Phi^T @ diag(weights)  and B = Phi
        Store A,B and the weights as torch tensors on device.
        """
        P, Q = Phi.shape[0], Phi.shape[1]
        Wdiag = (weights).astype(np.float64)   # P
        A = (Phi.T * Wdiag[None, :]).astype(np.float32)  # Q x P
        B = Phi.astype(np.float32)                        # P x Q

        self.A = torch.from_numpy(A).to(device)   # Q x P
        self.B = torch.from_numpy(B).to(device)   # P x Q
        self.weights = torch.from_numpy(weights.astype(np.float32)).to(device)  # (P,)
        self.P = P; self.Q = Q; self.device = device

    def grid_to_coeffs(self, x):
        """
        x: (B, C, H, W) torch tensor
        returns coeffs: (B, Q, C)
        vectorized: for each batch and each channel compute c = g @ A.T
        """
        Bsz, C, H, W = x.shape
        P = H * W
        x_flat = x.reshape(Bsz, C, P)           # (B, C, P)
        # multiply: (B, C, P) @ (P, Q) -> (B, C, Q)
        coeffs = torch.matmul(x_flat, self.A.t())  # (B, C, Q)
        coeffs = coeffs.permute(0, 2, 1).contiguous()  # (B, Q, C)
        return coeffs

    def coeffs_to_grid(self, coeffs, H, W):
        """
        coeffs: (B, Q, C) -> returns grid: (B, C, H, W)
        vectorized: for each batch and channel compute g = c @ B.T
        """
        Bsz, Q, C = coeffs.shape
        # permute to (B, C, Q)
        coeffs_pc = coeffs.permute(0, 2, 1).contiguous()  # (B, C, Q)
        # multiply: (B, C, Q) @ (Q, P) -> (B, C, P)
        grid_flat = torch.matmul(coeffs_pc, self.B.t())    # (B, C, P)
        grid = grid_flat.reshape(Bsz, C, H, W)
        return grid

In [ ]:
class SFNO(nn.Module):
    def __init__(self, transforms: SHTTransforms, index_to_l: np.ndarray, lmax: int,
                 in_ch=3, out_ch=3, hidden=64, n_blocks=3, device='cpu'):
        """
        transforms: SHTTransforms instance (contains A and B in torch)
        index_to_l: numpy array length Q mapping each coefficient index -> degree l
        lmax: maximum degree used
        hidden: hidden channel dimension after lift
        """
        super().__init__()
        self.transforms = transforms
        self.lmax = lmax
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.hidden = hidden
        self.n_blocks = n_blocks
        self.device = device

        # pointwise lift/proj
        self.input_proj = nn.Linear(in_ch, hidden)
        self.output_proj = nn.Linear(hidden, out_ch)

        # per-degree linear maps W_l (L+1, hidden, hidden) and biases b_l (L+1, hidden)
        self.W_l = nn.Parameter(torch.randn(lmax+1, hidden, hidden) * (1.0 / math.sqrt(hidden)))
        self.b_l = nn.Parameter(torch.zeros(lmax+1, hidden))

        # We'll vectorize application by expanding W_l -> (Q, hidden, hidden) according to index_to_l
        index_to_l_torch = torch.from_numpy(index_to_l.astype(np.int64)).to(device)  # (Q,)
        self.register_buffer('index_to_l', index_to_l_torch)  # buffer used to expand W per q

        # small pointwise MLP inside block
        self.pointwise = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU())

    def _expand_W_per_q(self):
        """
        Build W_expanded: (Q, hidden, hidden) by indexing self.W_l with index_to_l
        Also build b_expanded: (Q, hidden)
        This operation creates a tensor of size Q*hidden*hidden in memory.
        """
        # self.W_l: (L+1, hidden, hidden)
        # index_to_l: (Q,)
        W_expanded = self.W_l[self.index_to_l]    # (Q, hidden, hidden)
        b_expanded = self.b_l[self.index_to_l]    # (Q, hidden)
        return W_expanded, b_expanded

    def forward_block(self, x_hidden):
        """
        x_hidden: (B, hidden, H, W)
        Steps (vectorized):
         - grid -> coeffs: c (B, Q, hidden)
         - apply per-q linear maps: out_coeffs[b,q,k] = sum_h c[b,q,h] * W_expanded[q,h,k] + b_expanded[q,k]
         - coeffs -> grid: (B, hidden, H, W)
         - apply pointwise MLP and residual
        """
        Bsz, Hc, H, W = x_hidden.shape
        # grid -> coeffs
        coeffs = self.transforms.grid_to_coeffs(x_hidden)   # (B, Q, hidden)
        Q = coeffs.shape[1]
        # expand W_l per q (Q, hidden, hidden) and bias (Q, hidden)
        W_exp, b_exp = self._expand_W_per_q()               # (Q, h, h), (Q, h)
        # move W_exp to device (already on device since W_l is param)
        # Compute out_coeffs via einsum: (B, Q, h) x (Q, h, k) -> (B, Q, k)
        # We need matching dims: reorder W_exp: (Q, h, k)
        # Use torch.einsum with implicit broadcasting over Q:
        # coeffs: (B, Q, h); W_exp: (Q, h, k) -> out: (B, Q, k)
        out_coeffs = torch.einsum('bqh,qhk->bqk', coeffs, W_exp)  # (B, Q, hidden)
        out_coeffs = out_coeffs + b_exp.unsqueeze(0)              # broadcast bias over batch
        # coeffs -> grid
        out_grid = self.transforms.coeffs_to_grid(out_coeffs, H, W)  # (B, hidden, H, W)
        # pointwise: (B, hidden, H, W) -> (B, hidden, H, W)
        out_pw = out_grid.permute(0,2,3,1).contiguous().view(Bsz*H*W, self.hidden)
        out_pw = self.pointwise(out_pw)
        out_pw = out_pw.view(Bsz, H, W, self.hidden).permute(0,3,1,2)
        return x_hidden + out_pw  # residual

    def forward(self, x):
        """
        x: (B, in_ch, H, W)
        returns: (B, out_ch, H, W)
        """
        Bsz, Cin, H, W = x.shape
        # lift: pointwise linear
        x_pt = x.permute(0,2,3,1).contiguous()      # (B, H, W, in_ch)
        x_lift = self.input_proj(x_pt)              # (B, H, W, hidden)
        x_hidden = x_lift.permute(0,3,1,2).contiguous()  # (B, hidden, H, W)

        # n blocks


        for _ in range(self.n_blocks):
            x_hidden = self.forward_block(x_hidden)
        # project back
        x_back = x_hidden.permute(0,2,3,1).contiguous().view(Bsz*H*W, self.hidden)
        out_pt = self.output_proj(x_back)           # (B*H*W, out_ch)
        out = out_pt.view(Bsz, H, W, self.out_ch).permute(0,3,1,2).contiguous()
        return out

In [ ]:
class PlanetSWESingleSimDataset(Dataset):
    def __init__(
        self, 
        h5_path, 
        timesteps=1,
        max_timesteps=256,
        start_from=0
    ):
        """
        Args:
            h5_path (str): Путь к файлу.
            timesteps (int): Шаг прогноза (насколько шагов вперед смотрим).
            max_timesteps (int): Сколько временных шагов загрузить в память.
                                 Если None, загружает всю симуляцию (1008 кадров).
        """
        self.timesteps = timesteps
        self.start_from = start_from

        print(f"Opening {h5_path}...")
        with h5py.File(h5_path, 'r') as f:
            # Получаем полную длину времени в файле
            total_time = f['t0_fields/height'].shape[1]
            
            # Определяем, сколько будем читать
            self.T = total_time
            if max_timesteps is not None:
                self.T = min(total_time, max_timesteps + start_from)

            print(f"Loading {self.T} frames into RAM (from total {total_time})...")
            
            # Height: (1, T, H, W) -> [0, :T] -> (T, H, W) -> (T, H, W, 1)
            h = f['t0_fields/height'][0, :self.T]
            h = h[..., None] 

            # Velocity: (1, T, H, W, 2) -> [0, :T] -> (T, H, W, 2)
            v = f['t1_fields/velocity'][0, :self.T]\

            self.data = np.concatenate([h, v], axis=-1)

        print(f"Dataset ready. Shape in memory: {self.data.shape}")

    def __len__(self):
        return max(0, self.T - self.timesteps - self.start_from)

    def __getitem__(self, index):

        state_t = self.data[index + self.start_from]
        state_next = self.data[index + self.timesteps + self.start_from]

        # Конвертируем в Torch и меняем каналы местами (HWC -> CHW)
        x = torch.from_numpy(state_t).permute(2, 0, 1).float()
        y = torch.from_numpy(state_next).permute(2, 0, 1).float()

        return x, y

In [7]:
def relative_L2_torch(pred, target):
    # shapes (B,C,H,W)
    num = torch.sum((pred-target)**2, dim=[1,2,3])
    den = torch.sum(target**2, dim=[1,2,3]) + 1e-12
    return torch.sqrt(num/den)   # per-sample

def compute_mass_from_h(h_tensor, weights, H, W):
    """
    h_tensor: torch.Tensor shape (B, H, W)  (float)
    weights: torch.Tensor shape (P,)  (P = H*W) on same device
    Returns: mass tensor shape (B,)
    """
    B = h_tensor.shape[0]
    P = H * W
    w = weights.view(1, P)  # (1,P)
    h_flat = h_tensor.reshape(B, P)  # (B,P)
    mass = torch.sum(h_flat * w, dim=1)  # (B,)
    return mass


def train_sfno_with_mass(model, train_loader, val_loader, H, W,
                        epochs=10, lr=2e-4, alpha_M=0.1, eps=1e-12):
    """
    Loss = relative L2 (per-sample) + alpha_M * relative mass error (L1 per-sample)
    model.transforms.weights must be available (P,) on device.
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    device = model.device
    weights = model.transforms.weights  # (P,)
    for ep in tqdm(range(epochs), desc="Epochs"):
        model.train()
        running = 0.0
        for x, y in tqdm(train_loader, desc="train batches", leave=False):
            x = x.to(device); y = y.to(device)

            x = F.interpolate(x, size=(H, W), mode='bilinear', align_corners=False)
            y = F.interpolate(y, size=(H, W), mode='bilinear', align_corners=False)

            pred = model(x)  # (B, C, H, W)
            # --- relative L2 term ---
            num = torch.sum((pred - y)**2, dim=[1,2,3])  # (B,)
            den = torch.sum(y**2, dim=[1,2,3]) + eps     # (B,)
            relL2 = num / den                             # (B,)

            # --- mass term (use h channel = index 0) ---
            pred_h = pred[:, 0, :, :]   # (B,H,W)
            true_h = y[:, 0, :, :]      # (B,H,W)
            M_pred = compute_mass_from_h(pred_h, weights, H, W)  # (B,)
            M_true = compute_mass_from_h(true_h, weights, H, W)  # (B,)
            relM = torch.abs((M_pred - M_true) / (M_true + eps))  # (B,)

            loss_batch = torch.mean(relL2) + alpha_M * torch.mean(relM)

            opt.zero_grad()
            loss_batch.backward()
            opt.step()
            running += loss_batch.item()

        # validation (compute mean relative L2 and mass drift)
        model.eval()
        val_relL2_list = []
        val_relM_list = []
        with torch.no_grad():
            for x,y in tqdm(val_loader, desc="val batches", leave=False):
                x = x.to(device); y = y.to(device)

                x = F.interpolate(x, size=(H, W), mode='bilinear', align_corners=False)
                y = F.interpolate(y, size=(H, W), mode='bilinear', align_corners=False)

                pred = model(x)
                num = torch.sum((pred - y)**2, dim=[1,2,3])
                den = torch.sum(y**2, dim=[1,2,3]) + eps
                relL2 = (num/den).cpu().numpy()
                # mass
                pred_h = pred[:,0,:,:]; true_h = y[:,0,:,:]
                M_pred = compute_mass_from_h(pred_h, weights, H, W).cpu().numpy()
                M_true = compute_mass_from_h(true_h, weights, H, W).cpu().numpy()
                relM = np.abs((M_pred - M_true) / (M_true + eps))
                val_relL2_list.append(relL2)
                val_relM_list.append(relM)
        val_relL2 = np.mean(np.concatenate(val_relL2_list))
        val_relM = np.mean(np.concatenate(val_relM_list))
        print(f"Epoch {ep:02d}: train_loss={running/len(train_loader):.3e}, val_relL2={val_relL2:.3e}, val_relM={val_relM:.3e}")
    return model

In [ ]:
def demo(H=64,W=128,lmax=31,hidden=64,steps_pred=60,epochs=12, batch_size=16):
    device="cuda" if torch.cuda.is_available() else "cpu"

    print("building SHT basis...")
    Phi, w, index_to_l = build_real_sph_basis_Phi(H,W,lmax)
    transforms = SHTTransforms(Phi,w,device=device)

    train_ds = PlanetSWESingleSimDataset("./planetswe_IC00_s1.hdf5", max_timesteps=900)
    val_ds = PlanetSWESingleSimDataset("./planetswe_IC00_s1.hdf5",  max_timesteps=64, start_from=900)

    train_dl = DataLoader(train_ds,batch_size=batch_size,shuffle=True)
    val_dl = DataLoader(val_ds,batch_size=batch_size)


    model=SFNO(transforms,index_to_l,lmax,in_ch=3,out_ch=3,
                     hidden=hidden,n_blocks=3,device=device).to(device)

    print("Total parameters =", sum(p.numel() for p in model.parameters()) )


    model = train_sfno_with_mass(model, train_dl, val_dl, H=H, W=W,
                             epochs=epochs, lr=2e-4, alpha_M=0.1)


    print("Generating prediction rollout...")
    with torch.no_grad():
        x=train_ds[254][0].unsqueeze(0).to(device)

        x = F.interpolate(x, size=(H, W), mode='bilinear', align_corners=False)

        preds=[x.cpu().numpy()[0,0]] 

        for i in range(steps_pred):
            x=model(x)
            preds.append(x.cpu().numpy()[0,0])

    preds=np.array(preds)
    return train_ds[0], preds

In [10]:
arr, preds = demo(H=256,W=512,lmax=31,hidden=64,steps_pred=32,epochs=12,batch_size=16)
# visualize(arr, true_idx=256, preds=preds)

building SHT basis...


C:\Users\Nikita\AppData\Local\Temp\ipykernel_6716\147755526.py:36: DeprecationWarning: `scipy.special.sph_harm` is deprecated as of SciPy 1.15.0 and will be removed in SciPy 1.17.0. Please use `scipy.special.sph_harm_y` instead.
  Y_lm = sph_harm(m, l, lon_grid, theta_grid)  # complex HxW


Opening ./planetswe_IC00_s1.hdf5...
Loading 900 frames into RAM (from total 1008)...
Dataset ready. Shape in memory: (900, 256, 512, 3)
Opening ./planetswe_IC00_s1.hdf5...
Loading 964 frames into RAM (from total 1008)...
Dataset ready. Shape in memory: (964, 256, 512, 3)
Total parameters = 137731


Epochs:   8%|▊         | 1/12 [00:19<03:37, 19.79s/it]

Epoch 00: train_loss=1.589e+01, val_relL2=4.285e-01, val_relM=4.174e+01


...

Epoch 09: train_loss=4.505e+00, val_relL2=3.194e-01, val_relM=5.948e+01


Epochs:  92%|█████████▏| 11/12 [03:18<00:17, 17.84s/it]

Epoch 10: train_loss=3.508e+00, val_relL2=3.147e-01, val_relM=5.948e+01


Epochs: 100%|██████████| 12/12 [03:36<00:00, 18.07s/it]


Epoch 11: train_loss=3.834e+00, val_relL2=3.101e-01, val_relM=4.592e+01
Generating prediction rollout...


In [ ]:
#### visualization failed :(